## 1. Cài đặt thư viện

In [1]:
!pip install underthesea sklearn-crfsuite seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=96357eb936f44957867d3f9591cc48daafea9bdfe4be5d8fd8879e65be9322e5
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


## 2. Tiền xử lý và chuẩn bị dữ liệu

In [7]:
import os
from underthesea import ner

def extract_text_from_vlsp(filepath):
    """ Đọc file VLSP Sentiment và trích xuất câu văn thô """
    texts = []
    if not os.path.exists(filepath):
        return texts

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for i in range(len(lines)):
            if lines[i].startswith('#'):
                text = lines[i+1].strip()
                if text:
                    texts.append(text)
    return texts

def build_sequence_dataset(texts, max_sentences=500):
    dataset = []
    for text in texts[:max_sentences]:
        try:
            tagged_sentence = ner(text)
            sentence = [(word, pos, bio) for word, pos, chunk, bio in tagged_sentence]
            dataset.append(sentence)
        except:
            continue
    return dataset

# Đảm bảo bạn đã upload các file này lên môi trường Colab
train_file = '1-VLSP2018-SA-Restaurant-train (7-3-2018).txt'
test_file = '3-VLSP2018-SA-Restaurant-test (8-3-2018).txt'

raw_train = extract_text_from_vlsp(train_file)
raw_test = extract_text_from_vlsp(test_file)

if not raw_train:
    print("Không tìm thấy file txt, sử dụng dữ liệu giả lập...")
    raw_train = [
        "Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.",
        "Địa điểm thuận tiện, trong vòng bán kính 1,5km nhiều quán ăn ngon",
        "Hương vị thơm ngon, ăn cay cay rất thích. Quán ở Hà Nội."
    ]
    raw_test = [
        "Phòng ốc sạch, giường thoải mái. Gần Hồ Tây, view nhìn ra hồ lãng mạn"
    ]

print("Đang gán nhãn Sequence Labeling (Word, POS, NER)...")
train_sentences = build_sequence_dataset(raw_train, max_sentences=800)
test_sentences = build_sequence_dataset(raw_test, max_sentences=200)

print(f"Số câu tập Train: {len(train_sentences)}")
print(f"Số câu tập Test: {len(test_sentences)}")
print(f"Ví dụ: {train_sentences[0]}")

Đang gán nhãn Sequence Labeling (Word, POS, NER)...
Số câu tập Train: 800
Số câu tập Test: 200
Ví dụ: [('_Hương vị', 'N', 'O'), ('thơm', 'A', 'O'), ('ngon', 'A', 'O'), (',', 'CH', 'O'), ('ăn', 'V', 'O'), ('cay cay', 'N', 'O'), ('rất', 'R', 'O'), ('thích', 'V', 'O'), (',', 'CH', 'O'), ('nêm nếm', 'N', 'O'), ('vừa', 'R', 'O'), ('miệng', 'N', 'O'), ('.', 'CH', 'O'), ('Ngoài ra', 'X', 'O'), ('menu quán', 'N', 'O'), ('cũng', 'R', 'O'), ('nhiều', 'A', 'O'), ('món', 'N', 'O'), ('khác', 'A', 'O'), ('nhau', 'N', 'O'), ('tha hồ', 'V', 'O'), ('cho', 'E', 'O'), ('bạn', 'N', 'O'), ('lựa chọn', 'V', 'O'), ('luôn', 'R', 'O'), ('.', 'CH', 'O'), ('_Quán', 'N', 'O'), ('rộng rãi', 'A', 'O'), (',', 'CH', 'O'), ('view', 'N', 'O'), ('khá', 'R', 'O'), ('đẹp', 'A', 'O'), ('và', 'C', 'O'), ('cũng', 'R', 'O'), ('thoáng', 'V', 'O'), ('lắm', 'R', 'O'), ('.', 'CH', 'O'), ('Khách', 'N', 'O'), ('của', 'E', 'O'), ('quán', 'N', 'O'), ('đông', 'A', 'O'), ('nên', 'C', 'O'), ('nhiều', 'A', 'O'), ('khi', 'N', 'O'), ('nhân

## 3. Baseline Model

In [8]:
from collections import defaultdict, Counter

def train_baseline(train_sentences):
    word_pos_counts = defaultdict(Counter)
    for sentence in train_sentences:
        for word, pos, ner_tag in sentence:
            word_pos_counts[word][pos] += 1

    most_frequent_pos = {w: counts.most_common(1)[0][0] for w, counts in word_pos_counts.items()}
    return most_frequent_pos

def predict_baseline(test_sentences, pos_dict):
    y_pred_pos = []
    for sentence in test_sentences:
        pred_pos_sent = []
        for word, _, _ in sentence:
            pred_pos_sent.append(pos_dict.get(word, 'N')) # Fallback OOV là Danh từ
        y_pred_pos.append(pred_pos_sent)
    return y_pred_pos

baseline_pos_dict = train_baseline(train_sentences)
baseline_pos_pred = predict_baseline(test_sentences, baseline_pos_dict)

print("{:<15} | {:<10} | {:<10}".format("Word", "True POS", "Base POS"))
print("-" * 40)
for (word, true_pos, _), pred_pos in zip(test_sentences[0], baseline_pos_pred[0]):
    print("{:<15} | {:<10} | {:<10}".format(word, true_pos, pred_pos))

Đang gán nhãn Sequence Labeling (Word, POS, NER)...
Số câu tập Train: 800
Số câu tập Test: 200
Ví dụ: [('_Hương vị', 'N', 'O'), ('thơm', 'A', 'O'), ('ngon', 'A', 'O'), (',', 'CH', 'O'), ('ăn', 'V', 'O'), ('cay cay', 'N', 'O'), ('rất', 'R', 'O'), ('thích', 'V', 'O'), (',', 'CH', 'O'), ('nêm nếm', 'N', 'O'), ('vừa', 'R', 'O'), ('miệng', 'N', 'O'), ('.', 'CH', 'O'), ('Ngoài ra', 'X', 'O'), ('menu quán', 'N', 'O'), ('cũng', 'R', 'O'), ('nhiều', 'A', 'O'), ('món', 'N', 'O'), ('khác', 'A', 'O'), ('nhau', 'N', 'O'), ('tha hồ', 'V', 'O'), ('cho', 'E', 'O'), ('bạn', 'N', 'O'), ('lựa chọn', 'V', 'O'), ('luôn', 'R', 'O'), ('.', 'CH', 'O'), ('_Quán', 'N', 'O'), ('rộng rãi', 'A', 'O'), (',', 'CH', 'O'), ('view', 'N', 'O'), ('khá', 'R', 'O'), ('đẹp', 'A', 'O'), ('và', 'C', 'O'), ('cũng', 'R', 'O'), ('thoáng', 'V', 'O'), ('lắm', 'R', 'O'), ('.', 'CH', 'O'), ('Khách', 'N', 'O'), ('của', 'E', 'O'), ('quán', 'N', 'O'), ('đông', 'A', 'O'), ('nên', 'C', 'O'), ('nhiều', 'A', 'O'), ('khi', 'N', 'O'), ('nhân

## 4. Hidden Markov Model & Viterbi

In [9]:
import math

class HMMTagger:
    def __init__(self, k_smoothing=0.01):
        self.transitions = defaultdict(Counter)
        self.emissions = defaultdict(Counter)
        self.tag_counts = Counter()
        self.vocab = set()
        self.tags = set()
        self.k = k_smoothing # Tham số Laplace Smoothing

    def train(self, sentences):
        for sentence in sentences:
            prev_tag = '<s>'
            self.tag_counts['<s>'] += 1

            for word, tag, _ in sentence:
                self.vocab.add(word)
                self.tags.add(tag)
                self.transitions[prev_tag][tag] += 1
                self.emissions[tag][word] += 1
                self.tag_counts[tag] += 1
                prev_tag = tag

        # Áp dụng Laplace (Add-k) Smoothing cho Ma trận Transition (A)
        self.A = {}
        for prev_tag in self.tags | {'<s>'}:
            total_trans = sum(self.transitions[prev_tag].values())
            self.A[prev_tag] = {}
            for tag in self.tags:
                count = self.transitions[prev_tag][tag]
                self.A[prev_tag][tag] = math.log((count + self.k) / (total_trans + self.k * len(self.tags)))

        # Áp dụng Laplace Smoothing cho Ma trận Emission (B)
        self.B = {}
        V = len(self.vocab)
        for tag in self.tags:
            total_emis = sum(self.emissions[tag].values())
            self.B[tag] = {}
            for word in self.vocab:
                count = self.emissions[tag][word]
                self.B[tag][word] = math.log((count + self.k) / (total_emis + self.k * V))

            # Xác suất mặc định cho OOV (từ chưa biết)
            self.B[tag]['<UNK>'] = math.log(self.k / (total_emis + self.k * V))

    def viterbi_decode(self, sentence):
        if not sentence: return []
        viterbi = []
        backpointer = []

        # 1. Initialization (t=0)
        first_viterbi = {}
        first_backpointer = {}
        word = sentence[0]

        for state in self.tags:
            trans_prob = self.A.get('<s>', {}).get(state, math.log(1e-10))
            emis_prob = self.B.get(state, {}).get(word, self.B.get(state, {}).get('<UNK>', math.log(1e-10)))
            first_viterbi[state] = trans_prob + emis_prob
            first_backpointer[state] = 0

        viterbi.append(first_viterbi)
        backpointer.append(first_backpointer)

        # 2. Recursion (t > 0)
        for t in range(1, len(sentence)):
            word = sentence[t]
            curr_viterbi = {}
            curr_backpointer = {}

            for state in self.tags:
                max_prob = -float('inf')
                best_prev_state = None
                emis_prob = self.B.get(state, {}).get(word, self.B.get(state, {}).get('<UNK>', math.log(1e-10)))

                for prev_state in self.tags:
                    trans_prob = self.A.get(prev_state, {}).get(state, math.log(1e-10))
                    prob = viterbi[t-1][prev_state] + trans_prob + emis_prob

                    if prob > max_prob:
                        max_prob = prob
                        best_prev_state = prev_state

                curr_viterbi[state] = max_prob
                curr_backpointer[state] = best_prev_state

            viterbi.append(curr_viterbi)
            backpointer.append(curr_backpointer)

        # 3. Termination
        best_last_state = max(viterbi[-1].keys(), key=lambda s: viterbi[-1][s])
        best_path = [best_last_state]
        for t in range(len(sentence) - 1, 0, -1):
            best_path.insert(0, backpointer[t][best_path[0]])

        return best_path

hmm_pos = HMMTagger()
hmm_pos.train(train_sentences)

test_words = [t[0] for t in test_sentences[0]]
pred_pos_hmm = hmm_pos.viterbi_decode(test_words)

print("{:<15} | {:<10}".format("Word", "HMM POS"))
print("-" * 30)
for w, tag in zip(test_words, pred_pos_hmm):
    print("{:<15} | {:<10}".format(w, tag))

Word            | HMM POS   
------------------------------
Bữa             | N         
lướt            | P         
facebook        | Z         
nhỏ             | A         
bạn             | N         
thấy            | V         
có              | V         
check-in        | L         
ly              | N         
trà             | N         
sữa             | N         
khổng lồ        | A         
cùng viên       | CH        
phô mai         | N         
to              | A         
ú nụ            | I         
.               | CH        
Mình            | P         
bị              | V         
hấp dẫn         | V         
trong           | E         
ngày            | N         
từ              | E         
cái             | Nc        
nhìn            | V         
đầu tiên        | A         
nên             | C         
đòi             | L         
anh             | Nc        
người yêu       | N         
chở             | V         
đi              | V         
mua         

## 5. Conditional Random Fields (CRF) cho bài toán NER

In [10]:
import sklearn_crfsuite

def word2features(sent, i):
    word = sent[i][0]
    pos = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[:3]': word[:3],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'word.length()': len(word),
        'postag': pos,
    }

    if i > 0:
        word1 = sent[i-1][0]
        pos1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:postag': pos1,
        })
    else:
        features['BOS'] = True

    if i < len(sent) - 1:
        word1 = sent[i+1][0]
        pos1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:postag': pos1,
        })
    else:
        features['EOS'] = True

    return features

def extract_features(sentences):
    return [[word2features(s, i) for i in range(len(s))] for s in sentences]

def extract_ner_labels(sentences):
    return [[token[2] for token in s] for s in sentences]

X_train = extract_features(train_sentences)
y_train_ner = extract_ner_labels(train_sentences)
X_test = extract_features(test_sentences)
y_test_ner = extract_ner_labels(test_sentences)

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train_ner)

y_pred_ner = crf.predict(X_test)

print("{:<15} | {:<10} | {:<10}".format("Word", "True NER", "CRF NER"))
print("-" * 40)
for (w, _, true_tag), pred_tag in zip(test_sentences[0], y_pred_ner[0]):
    print("{:<15} | {:<10} | {:<10}".format(w, true_tag, pred_tag))

Word            | True NER   | CRF NER   
----------------------------------------
Bữa             | O          | O         
lướt            | O          | O         
facebook        | O          | O         
nhỏ             | O          | O         
bạn             | O          | O         
thấy            | O          | O         
có              | O          | O         
check-in        | O          | O         
ly              | O          | O         
trà             | O          | O         
sữa             | O          | O         
khổng lồ        | O          | O         
cùng viên       | O          | O         
phô mai         | O          | O         
to              | O          | O         
ú nụ            | O          | O         
.               | O          | O         
Mình            | O          | O         
bị              | O          | O         
hấp dẫn         | O          | O         
trong           | O          | O         
ngày            | O          | O   

## 6. Đánh giá NER

In [13]:
from seqeval.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

print("=== Báo cáo Đánh giá (CRF NER) trên tập dữ liệu Test ===")
# Lưu ý: Hiệu năng phụ thuộc vào chất lượng của bộ nhãn được gen tự động từ thư viện underthesea
print(classification_report(y_test_ner, y_pred_ner))

=== Báo cáo Đánh giá (CRF NER) trên tập dữ liệu Test ===
              precision    recall  f1-score   support

         LOC       0.84      0.67      0.74       492
        MISC       0.20      0.11      0.14         9
         ORG       0.00      0.00      0.00         7
         PER       0.81      0.93      0.87       300

   micro avg       0.82      0.75      0.79       808
   macro avg       0.46      0.43      0.44       808
weighted avg       0.81      0.75      0.78       808



In [15]:
from seqeval.metrics import classification_report
from underthesea import ner
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. ĐÁNH GIÁ POS TAGGING (BẰNG ACCURACY)
# ==========================================
total_tokens = 0
correct_pos = 0

for i in range(len(test_sentences)):
    test_words = [token[0] for token in test_sentences[i]]
    true_pos = [token[1] for token in test_sentences[i]]
    # Sử dụng HMM đã train ở Ô Code 4 để dự đoán
    pred_pos = hmm_pos.viterbi_decode(test_words)

    total_tokens += len(true_pos)
    correct_pos += sum(1 for t, p in zip(true_pos, pred_pos) if t == p)

print("=== ĐÁNH GIÁ MÔ HÌNH HMM (POS TAGGING) ===")
print(f"Độ chính xác (Accuracy) trên toàn bộ tập Test: {correct_pos / total_tokens * 100:.2f}%\n")

=== ĐÁNH GIÁ MÔ HÌNH HMM (POS TAGGING) ===
Độ chính xác (Accuracy) trên toàn bộ tập Test: 83.70%



In [16]:
# ==========================================
# 2. ĐÁNH GIÁ CRF NER (DEMO VỚI CÂU THỰC TẾ)
# ==========================================
print("=== ĐÁNH GIÁ MÔ HÌNH CRF (NER) ===")
print("Do tập review Nhà hàng/Khách sạn gần như không chứa tên riêng, bảng báo cáo seqeval gốc sẽ có Support = 0.")
print("Dưới đây là Demo mô hình CRF nhận diện trên một câu có chứa Thực thể:\n")

# Câu test tự tạo chứa địa danh, tên tổ chức, tên người
sample_text = "Trường học Đại học Bách Khoa ở Thủ đô Hà Nội có thầy giáo tên là Nam rất nhiệt tình."
sample_sent = [(word, pos, bio) for word, pos, chunk, bio in ner(sample_text)]

# Trích xuất đặc trưng và dự đoán bằng CRF đã train ở Ô Code 5
sample_X = extract_features([sample_sent])
sample_pred = crf.predict(sample_X)

print("{:<20} | {:<10} | {:<10}".format("Word", "True NER", "CRF NER"))
print("-" * 45)
for (w, _, true_tag), pred_tag in zip(sample_sent, sample_pred[0]):
    print("{:<20} | {:<10} | {:<10}".format(w, true_tag, pred_tag))

=== ĐÁNH GIÁ MÔ HÌNH CRF (NER) ===
Do tập review Nhà hàng/Khách sạn gần như không chứa tên riêng, bảng báo cáo seqeval gốc sẽ có Support = 0.
Dưới đây là Demo mô hình CRF nhận diện trên một câu có chứa Thực thể:

Word                 | True NER   | CRF NER   
---------------------------------------------
Trường học           | O          | O         
Đại học              | B-LOC      | B-LOC     
Bách Khoa            | I-LOC      | I-LOC     
ở                    | O          | O         
Thủ đô               | O          | B-LOC     
Hà Nội               | B-LOC      | I-LOC     
có                   | O          | O         
thầy giáo            | O          | O         
tên                  | O          | O         
là                   | O          | O         
Nam                  | B-PER      | B-PER     
rất                  | O          | O         
nhiệt tình           | O          | O         
.                    | O          | O         


In [19]:
from underthesea import ner

# 1. Hiển thị ô nhập liệu cho người dùng
user_input = input("Nhập câu bạn muốn kiểm thử: ")

if user_input.strip() == "":
    print("Bạn chưa nhập câu nào!")
else:
    # 2. Tiền xử lý: Dùng underthesea để tách từ và lấy nhãn POS nền tảng
    # Output của ner(text) là danh sách các tuple: (word, pos, chunk, ner)
    tagged_input = ner(user_input)

    # Lấy danh sách các từ để đưa vào HMM
    words = [word for word, pos, chunk, bio in tagged_input]

    # 3. DỰ ĐOÁN POS BẰNG HMM (Đã train ở Ô Code 4)
    pred_pos_hmm = hmm_pos.viterbi_decode(words)

    # 4. DỰ ĐOÁN NER BẰNG CRF (Đã train ở Ô Code 5)
    # Tạo cấu trúc câu giả định [(Word, POS, 'O')] để đưa vào hàm extract_features
    mock_sentence = [(word, pos, 'O') for word, pos, chunk, bio in tagged_input]

    # Trích xuất đặc trưng và dự đoán
    X_user = extract_features([mock_sentence])
    pred_ner_crf = crf.predict(X_user)[0]

    # 5. IN KẾT QUẢ ĐẸP MẮT
    print("\n" + "="*50)
    print("KẾT QUẢ DỰ ĐOÁN SEQUENCE LABELING")
    print("="*50)
    print("{:<20} | {:<12} | {:<12}".format("Từ (Word)", "HMM (POS)", "CRF (NER)"))
    print("-" * 50)

    for w, hmm_pos_tag, crf_ner_tag in zip(words, pred_pos_hmm, pred_ner_crf):
        print("{:<20} | {:<12} | {:<12}".format(w, hmm_pos_tag, crf_ner_tag))

Nhập câu bạn muốn kiểm thử: Tôi là Cáp Kim Hải Anh, học tại Trường Đại học Công nghệ thông tin - ĐHQG-HCM

KẾT QUẢ DỰ ĐOÁN SEQUENCE LABELING
Từ (Word)            | HMM (POS)    | CRF (NER)   
--------------------------------------------------
Tôi                  | P            | O           
là                   | V            | O           
Cáp Kim Hải Anh      | T            | B-LOC       
,                    | CH           | O           
học                  | V            | O           
tại                  | E            | O           
Trường               | L            | B-LOC       
Đại học              | N            | I-LOC       
Công nghệ thông tin  | I            | I-LOC       
-                    | CH           | I-LOC       
ĐHQG-HCM             | X            | I-LOC       
